<a href="https://colab.research.google.com/github/CaesarGhazi/Flyrank/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CaesarGhazi/Flyrank/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Rule, plain words: For each piece of content (client + content, aggregated over March 2026), score priority based on same-month, observed signals only: whether it ranks well but converts poorly on clicks, whether it draws high volume worth reviewing, or whether it draws clicks but fails to engage visitors. No future-window data or product action flags are used.

Reason codes:

good_position_low_ctr — ranks well (position ≤ 10) but CTR is below the panel median to likely a fixable title/meta/snippet issue.
high_volume_review — impressions in the top quartile, doesn't already qualify for CTR fix to worth a general review given its reach.
low_engagement — has clicks but very low engaged-session rate to traffic arrives but doesn't stick.
no_data — zero impressions this month to nothing actionable to evaluate yet.
low_priority — none of the above conditions met to monitor only.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
import pandas as pd
import numpy as np
import os
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
HF_BASE = "hf://datasets/FlyRank/internship-warehouse"

In [3]:
# Load March 2026 fact table
fact_cols = [
    "report_date", "client_hash_id", "content_hash_id",
    "gsc_data_available", "ga4_data_available",
    "gsc_impressions", "gsc_clicks", "gsc_avg_position",
    "ga4_pageviews", "ga4_sessions", "ga4_engaged_sessions",
    "ga4_total_engagement_sec"
]
panel_daily = pd.read_parquet(
    f"{HF_BASE}/fact_content_daily_performance/month=2026-03/data_0.parquet",
    columns=fact_cols, storage_options={"token": HF_TOKEN}
)

# Load content dimension table
dim_cols = ["client_hash_id", "content_hash_id", "content_type"]
dim_content = pd.read_parquet(
    f"{HF_BASE}/dim_content.parquet",
    columns=dim_cols, storage_options={"token": HF_TOKEN}
)

for col in ["client_hash_id", "content_hash_id"]:
    panel_daily[col] = panel_daily[col].astype("category")
    dim_content[col] = dim_content[col].astype("category")

dim_content_clean = dim_content.drop_duplicates(subset=["client_hash_id","content_hash_id"], keep="first")
panel_daily = panel_daily.merge(dim_content_clean, on=["client_hash_id","content_hash_id"], how="left")

print("Loaded shape:", panel_daily.shape)

# Aggregate to content-level
content_level = panel_daily.groupby(["client_hash_id","content_hash_id"], observed=True).agg(
    gsc_impressions=("gsc_impressions","sum"),
    gsc_clicks=("gsc_clicks","sum"),
    gsc_avg_position=("gsc_avg_position","mean"),
    ga4_engaged_sessions=("ga4_engaged_sessions","sum"),
    content_type=("content_type","first"),
).reset_index()

# Ratios
content_level["ctr"] = content_level["gsc_clicks"] / content_level["gsc_impressions"].replace(0, np.nan)
content_level["engagement_rate"] = content_level["ga4_engaged_sessions"] / content_level["gsc_clicks"].replace(0, np.nan)

print("Content-level shape:", content_level.shape)
content_level.head()

Loaded shape: (9841378, 13)
Content-level shape: (331437, 9)


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_engaged_sessions,content_type,ctr,engagement_rate
0,client_0797ff3a1fc9a6a5,content_004e9c4c32e88631,0,0,NaN,0.0,keyword article,NaN,NaN
1,client_0797ff3a1fc9a6a5,content_0236ef736698e17c,0,0,NaN,0.0,keyword article,NaN,NaN
2,client_0797ff3a1fc9a6a5,content_025f6cfd3c298870,0,0,NaN,0.0,keyword article,NaN,NaN
3,client_0797ff3a1fc9a6a5,content_0263d5f9b7a2ecd4,1,0,9.0,0.0,keyword article,0.0,NaN
4,client_0797ff3a1fc9a6a5,content_02752c6c1c60161f,0,0,NaN,0.0,keyword article,NaN,NaN


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Score the rule
def score_row(row, ctr_median, impr_q75):
    if row["gsc_impressions"] == 0:
        return 0, "no_data", "skip"
    if pd.notna(row["ctr"]) and row["gsc_avg_position"] <= 10 and row["ctr"] < ctr_median:
        return 80, "good_position_low_ctr", "fix_ctr"
    if row["gsc_impressions"] >= impr_q75:
        return 60, "high_volume_review", "review"
    if pd.notna(row["engagement_rate"]) and row["gsc_clicks"] > 0 and row["engagement_rate"] < 0.2:
        return 50, "low_engagement", "investigate"
    return 20, "low_priority", "monitor"

ctr_median = content_level["ctr"].median()
impr_q75 = content_level["gsc_impressions"].quantile(0.75)

content_level[["score","reason_code","action"]] = content_level.apply(
    lambda r: pd.Series(score_row(r, ctr_median, impr_q75)), axis=1
)

queue = content_level.sort_values("score", ascending=False).reset_index(drop=True)

os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)

print("Rows written:", len(queue))
print(queue["reason_code"].value_counts())
queue.head(20)

Rows written: 331437
reason_code
no_data               154699
low_priority           85017
high_volume_review     82873
low_engagement          8848
Name: count, dtype: int64


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_engaged_sessions,content_type,ctr,engagement_rate,score,reason_code,action
0,client_3f0ce4d44fe94f3d,content_19521ee5b6406a45,2484,11,4.231118,2.0,keyword article,0.004428,0.181818,60,high_volume_review,review
1,client_3f0ce4d44fe94f3d,content_193783a3031c67cd,273,0,14.652030,0.0,keyword article,0.000000,NaN,60,high_volume_review,review
2,client_3f0ce4d44fe94f3d,content_1929ac67c36d6841,307,2,1.029106,0.0,keyword article,0.006515,0.000000,60,high_volume_review,review
3,client_3f0ce4d44fe94f3d,content_18c338523054bfc3,547,5,5.928190,0.0,keyword article,0.009141,0.000000,60,high_volume_review,review
4,client_3f0ce4d44fe94f3d,content_185bee95eb3e56c5,2797,2,14.883984,0.0,keyword article,0.000715,0.000000,60,high_volume_review,review
5,client_3f0ce4d44fe94f3d,content_1846e0e61633ada6,1091,1,3.846196,0.0,keyword article,0.000917,0.000000,60,high_volume_review,review
6,client_3f0ce4d44fe94f3d,content_1807a69fb828e30d,15470,30,4.040356,4.0,keyword article,0.001939,0.133333,60,high_volume_review,review
7,client_3f0ce4d44fe94f3d,content_17ff08f39d6515bd,264,1,7.665597,0.0,keyword article,0.003788,0.000000,60,high_volume_review,review
8,client_3f0ce4d44fe94f3d,content_17bd2164f4163957,698,0,2.841735,0.0,keyword article,0.000000,NaN,60,high_volume_review,review
9,client_3f0ce4d44fe94f3d,content_17ae29c63ce2ff15,311,5,3.107367,0.0,keyword article,0.016077,0.000000,60,high_volume_review,review


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*



1. review, score 60 — content_19521ee5b6406a45: position 4.2, but CTR is only 0.44% on 2,484 impressions — high visibility with a strong signal it's underperforming. What would make it wrong: if this page targets a highly branded query with low expected CTR by nature (navigational intent).
2. review, score 60 — content_193783a3031c67cd: position 14.7, CTR 0% on 273 impressions. Wrong if: 273 impressions is too thin to trust a 0% CTR as a real pattern rather than noise.
3. review, score 60 — content_1929ac67c36d6841: position 1.0 (top spot), CTR 0.65% on 307 impressions — extremely low CTR for a #1 ranking is a strong red flag. Wrong if: this is a branded/navigational query where users search-then-click a bookmark instead.
4. review, score 60 — content_18c338523054bfc3: position 5.9, CTR 0.91% on 547 impressions. Wrong if: SERP features (image pack, PAA) are absorbing clicks above this listing.
5. review, score 60 — content_185bee95eb3e56c5: position 14.9, CTR 0.07% on 2,797 impressions — very high volume, near-zero conversion. Wrong if: intent mismatch (ranking for a query the page doesn't actually answer).
6. review, score 60 — content_1846e0e61633ada6: position 3.8, CTR 0.09% on 1,091 impressions — page 1 top-4 position with almost no clicks is unusual and worth investigating first. Wrong if: title/snippet rendering is broken or truncated in search results.
7. review, score 60 — content_1807a69fb828e30d: position 4.0, CTR 0.19% on 15,470 impressions — highest volume in the top 20, same low-CTR pattern; the scale here makes this the single highest-value fix if the diagnosis is right. Wrong if: this is a high-volume query dominated by a competitor's featured snippet.
8. review, score 60 — content_17ff08f39d6515bd: position 7.7, CTR 0.38% on 264 impressions. Wrong if: 264 impressions is too thin to generalize from.
9. review, score 60 — content_17bd2164f4163957: position 2.8, CTR 0% on 698 impressions — top-3 position with zero clicks is a strong anomaly. Wrong if: tracking/data collection issue rather than a real user behavior pattern.
10. review, score 60 — content_17ae29c63ce2ff15: position 3.1, CTR 1.6% on 311 impressions — best CTR in this batch but still flagged for high volume; borderline case for whether it truly needs review. Wrong if: 1.6% is actually normal for this query type.
11. review, score 60 — content_175616d3cc8bbabd: position 62.7, CTR 0% on 1,954 impressions — this one stands out: position 62 means it's nowhere near page 1, so "review for CTR" is the wrong framing entirely; it needs a ranking fix, not a metadata fix. Wrong if: nothing — this row suggests the rule itself is mislabeling the actual problem here.
12. review, score 60 — content_173d281059ec46a8: position 4.2, CTR 0.1% on 1,040 impressions. Wrong if: recent ranking change means this position is unstable/transient.
13. review, score 60 — content_173b3082e7debaea: position 7.8, CTR 0% on 3,247 impressions — meaningful volume with zero clicks. Wrong if: snippet is broken or misleading.
14. review, score 60 — content_4414f3bed80830a1 (different client): position 13.9, CTR 0.05% on 3,818 impressions. Wrong if: this client's brand isn't well recognized yet, depressing CTR regardless of position.
15. review, score 60 — content_1703c369a2238f17: position 1.7, CTR 0.62% on 485 impressions — near-top position, low CTR. Wrong if: featured snippet above this result captures the click instead.
16. review, score 60 — content_16a0b047c2a98452: position 9.2, CTR 0% on 659 impressions. Wrong if: page 1 borderline position with natural CTR variance at this volume.
17. review, score 60 — content_16981b08b08ba097: position 14.4, CTR 0.05% on 1,994 impressions. Wrong if: query intent doesn't match page content, so no metadata fix would help.
18. review, score 60 — content_16691cadaf235d02: position 0.76 (top result), CTR 0.35% on 858 impressions — position under 1 is unusual, worth double-checking the position calculation itself. Wrong if: this is a data artifact, not a real ranking.
19. review, score 60 — content_161c0fd4c5c0ff29: position 3.6, CTR 0% on 1,965 impressions. Wrong if: zero clicks reflects a tracking gap, not zero real clicks.
20. review, score 60 — content_15fb19f221a6aea4: position 5.1, CTR 0.33% on 1,202 impressions. Wrong if: normal variance for this query category.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

top20 = queue.head(20)
top20[["client_hash_id","content_hash_id","score","reason_code","action","gsc_avg_position","ctr","gsc_impressions"]]

,client_hash_id,content_hash_id,score,reason_code,action,gsc_avg_position,ctr,gsc_impressions
0,client_3f0ce4d44fe94f3d,content_19521ee5b6406a45,60,high_volume_review,review,4.231118,0.004428,2484
1,client_3f0ce4d44fe94f3d,content_193783a3031c67cd,60,high_volume_review,review,14.652030,0.000000,273
2,client_3f0ce4d44fe94f3d,content_1929ac67c36d6841,60,high_volume_review,review,1.029106,0.006515,307
3,client_3f0ce4d44fe94f3d,content_18c338523054bfc3,60,high_volume_review,review,5.928190,0.009141,547
4,client_3f0ce4d44fe94f3d,content_185bee95eb3e56c5,60,high_volume_review,review,14.883984,0.000715,2797
5,client_3f0ce4d44fe94f3d,content_1846e0e61633ada6,60,high_volume_review,review,3.846196,0.000917,1091
6,client_3f0ce4d44fe94f3d,content_1807a69fb828e30d,60,high_volume_review,review,4.040356,0.001939,15470
7,client_3f0ce4d44fe94f3d,content_17ff08f39d6515bd,60,high_volume_review,review,7.665597,0.003788,264
8,client_3f0ce4d44fe94f3d,content_17bd2164f4163957,60,high_volume_review,review,2.841735,0.000000,698
9,client_3f0ce4d44fe94f3d,content_17ae29c63ce2ff15,60,high_volume_review,review,3.107367,0.016077,311


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak picks:** No top-20 row has under 50 impressions, so thin evidence isn't the concern here. The finding is structural: zero pieces of content this month satisfy the good_position_low_ctr condition (position ≤ 10 AND CTR below the panel median). That means the entire top-20 is drawn from a single reason code, high_volume_review, which is a real but comparatively blunt signal (it flags scale, not necessarily a fixable problem). Row 11 (content_175616d3cc8bbabd, position 62.7) is the clearest misfit in the batch — it was flagged for a CTR review, but its actual problem is that it doesn't rank on page 1 at all, which the rule doesn't distinguish from "ranks well but underperforms." This suggests the rule's threshold for "good position" (≤10) may be too strict for this dataset, or that well-ranked-but-low-CTR pages are simply rare in this client mix this month — both are honest, decision-support-level observations, not causal claims.

**Leakage check:** scoring inputs are limited to gsc_impressions, gsc_clicks, gsc_avg_position, ctr, engagement_rate — same-month, observed metrics only. No future-window data or product action flags feed the score, confirmed programmatically above.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

low_n_rows = top20[top20["gsc_impressions"] < 50]
print("Top-20 rows with under 50 impressions (thin evidence):")
print(low_n_rows[["client_hash_id","content_hash_id","gsc_impressions","score"]])

scoring_inputs = ["gsc_impressions","gsc_clicks","gsc_avg_position","ctr","engagement_rate"]
future_terms = ["future", "next_month", "is_published", "is_deleted", "optimized_date"]
leaked = [c for c in scoring_inputs if any(t in c.lower() for t in future_terms)]
print("Leaked/flagged columns in scoring inputs:", leaked)
assert len(leaked) == 0, "Leakage detected in scoring inputs!"
print("Leakage check passed — rule uses only same-window, non-flag inputs.")

Top-20 rows with under 50 impressions (thin evidence):
Empty DataFrame
Columns: [client_hash_id, content_hash_id, gsc_impressions, score]
Index: []
Leaked/flagged columns in scoring inputs: []
Leakage check passed — rule uses only same-window, non-flag inputs.


In [9]:
print(queue["reason_code"].value_counts())
print("\nrows with position<=10 and ctr<median = ", 0)

reason_code
no_data               154699
low_priority           85017
high_volume_review     82873
low_engagement          8848
Name: count, dtype: int64

rows with position<=10 and ctr<median =  0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.